# HDC Cognitive Architecture v3 — runnable prototype

Этот notebook запускает проверяемое ядро архитектуры без API, ключей и платных моделей. Он демонстрирует HDC-ассоциации, динамический семантический слой, STM/LTM, регуляторы, консолидацию, rollback и сравнение с n-граммным baseline.

**Важно:** это эксперимент с ассоциативной памятью, а не доказательство сознания или понимания языка. Нажмите **Runtime → Run all**.

In [ ]:
# 1. Загрузка реализации — дополнительных пакетов не требуется
import pathlib, sys, urllib.request

# Пин на проверенную benchmark-версию; Colab не получит случайное будущее изменение.
MODULE_URL = 'https://raw.githubusercontent.com/marieabdlk-art/razlom4/a423be2/hdc_cognitive.py'
module_path = pathlib.Path('/content/hdc_cognitive.py')
urllib.request.urlretrieve(MODULE_URL, module_path)
sys.path.insert(0, '/content')

from hdc_cognitive import (
    HDCCognitiveSystem, NGramBaseline, BackoffNGramBaseline,
    CharNGramKNNBaseline, evaluate_next_token,
    synthetic_language, tokenize
)
print('✓ HDC Cognitive v3 загружен')

In [ ]:
# 2. Быстрое обучение на знакомом фрагменте
pushkin = 'У лукоморья дуб зелёный, златая цепь на дубе том.'
model = HDCCognitiveSystem(
    dimension=2048, seed='pushkin-demo',
    stm_capacity=512, ltm_capacity=2048
)
result = model.train_sequences([pushkin] * 8)
print('После обучения:', result)

query = ['у', 'лукоморья', 'дуб']
prediction = model.predict(query)
print('Контекст:', query)
print('Предсказание:', prediction)
print('Продолжение:', ' '.join(model.generate('у лукоморья', 12)))

In [ ]:
# 3. Честный held-out тест: пять равнобюджетных вариантов
train_rows, test_rows = synthetic_language(seed=7, train_size=300, test_size=100)

common = dict(dimension=2048, seed='blind-demo', stm_capacity=2048, ltm_capacity=4096)
arms = {
    'Exact n-gram': NGramBaseline(context_length=4, max_contexts=2048),
    'Backoff n-gram': BackoffNGramBaseline(context_length=4, max_contexts=2048),
    'Character KNN': CharNGramKNNBaseline(context_length=4, max_contexts=2048),
    'Static HDC': HDCCognitiveSystem(dynamic_semantics=False, **common),
    'Dynamic HDC': HDCCognitiveSystem(dynamic_semantics=True, **common),
}
scores = {}
for name, arm in arms.items():
    arm.train_sequences(train_rows)
    scores[name] = evaluate_next_token(arm, test_rows)
    print(f'{name:16s}', scores[name])

winner = max(scores, key=lambda name: scores[name]['accuracy'])
print('Победитель этого маленького чистого прогона:', winner)
hdc = arms['Static HDC']

In [ ]:
# 4. Графики точности и покрытия
try:
    import matplotlib.pyplot as plt
    labels = list(scores)
    accuracy = [scores[name]['accuracy'] for name in labels]
    coverage = [scores[name]['coverage'] for name in labels]
    x = range(len(labels))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(x, accuracy, color=['#6750A4', '#777777'])
    axes[0].set_title('Held-out accuracy')
    axes[0].set_xticks(list(x), labels, rotation=30, ha='right')
    axes[0].set_ylim(0, 1)
    axes[1].bar(x, coverage, color=['#6750A4', '#777777'])
    axes[1].set_title('Coverage')
    axes[1].set_xticks(list(x), labels, rotation=30, ha='right')
    axes[1].set_ylim(0, 1)
    plt.show()
except ImportError:
    print('matplotlib отсутствует; численные результаты показаны выше')

In [ ]:
# 5. STM → LTM, консолидация и точный rollback
hash_before = hdc.behavior_hash()
summary_before = hdc.summary()
consolidated = hdc.consolidate()
hash_after = hdc.behavior_hash()
rollback = hdc.rollback()
hash_restored = hdc.behavior_hash()

print('До:', summary_before)
print('Консолидация:', consolidated)
print('Состояние изменилось:', hash_before != hash_after)
print('Rollback:', rollback['status'])
print('Поведение восстановлено побитово:', hash_before == hash_restored)
assert hash_before == hash_restored

In [ ]:
# 6. Проверка воспроизводимости: одинаковый seed + данные = одинаковое состояние
left = HDCCognitiveSystem(dimension=1024, seed='replay-check')
right = HDCCognitiveSystem(dimension=1024, seed='replay-check')
sample = ['кот видит ключ .', 'лиса ищет книгу .', 'робот несёт мяч .'] * 4
left.train_sequences(sample)
right.train_sequences(sample)
print('Hash A:', left.behavior_hash())
print('Hash B:', right.behavior_hash())
print('Детерминировано:', left.behavior_hash() == right.behavior_hash())
assert left.behavior_hash() == right.behavior_hash()

In [ ]:
# 7. Ваш собственный текст
MY_TEXT = '''
Маленький робот нашёл красный ключ.
Красный ключ открывает старую дверь.
За старой дверью находится тихий сад.
'''
custom = HDCCognitiveSystem(dimension=2048, seed='my-text')
custom.train_sequences([MY_TEXT] * 10)
print('Продолжение:', ' '.join(custom.generate('красный ключ', 20)))
print('Состояние:', custom.summary())

## Как трактовать результат

- Если HDC запоминает знакомый фрагмент — работает ассоциативная память.
- Если HDC обходит сильные backoff/character controls на held-out комбинациях — есть предварительный сигнал полезного обобщения.
- Один маленький прогон ничего не доказывает: нужны разные seeds, большие корпуса и доверительные интервалы.
- В зафиксированном десяти-seed тесте HDC-ядро оказалось полезным на шумных и порядковых задачах, но dynamic semantic overlay не превзошёл static HDC с учётом памяти. Поэтому static HDC используется по умолчанию.
- Гормоны и когерентность в прототипе являются ограниченными инженерными регуляторами, а не моделью человеческой психики.